# 2025 홀드아웃 통합 피처 엔지니어링 (학습데이터와 결합)

**전제**: 2025 홀드아웃에 1단계 위험점수 결합은 이미 완료된 상태 (`2단계_2025홀드아웃_1단계포함.csv.gz`)

**이 노트북이 하는 일**
1. 학습데이터(58컬럼) + 1단계 결합된 2025 홀드아웃(96컬럼) 로드
2. 각각 라벨 보정(`최종_보정라벨`) 계산 (동일 로직, 독립적으로 가능)
3. 홀드아웃만 원-핫 인코딩(매칭등급, 1단계_대체수준) - 학습데이터는 이미 되어있음
4. 홀드아웃을 학습데이터와 같은 컬럼 구조로 정렬
5. **둘을 하나로 합쳐서** 신규 피처 10개(누적타겟인코딩/diff4/roll_std_4q/계절성/상대비율) 계산
   (합쳐야 하는 이유: 2025 데이터의 "4분기 전 값", "누적 폐업률"을 계산하려면 2022~2024년 이력이 필요)
6. 다시 학습/홀드아웃으로 분리해서 각각 저장

모든 경로는 `/content/drive/MyDrive/kb_challenge` 기준입니다.

In [1]:
import os
import gc
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/kb_challenge'

TRAIN_PATH = os.path.join(DRIVE_DIR, '2단계_학습데이터_최종_인허가포함.parquet')
HOLDOUT_PATH = os.path.join(DRIVE_DIR, '2단계_2025홀드아웃_1단계포함.parquet')

TRAIN_OUTPUT = os.path.join(DRIVE_DIR, '2단계_학습데이터_최종_인허가포함_피처완료.parquet')
HOLDOUT_OUTPUT = os.path.join(DRIVE_DIR, '2단계_2025홀드아웃_피처완료.parquet')

ORIGINAL_TARGET_COL = '폐업_다음분기'
TARGET_COL = '최종_보정라벨'
QUARTER_COL = '기준_년분기'
ID_COL = '상가업소번호'


## 1. 로드 + 컬럼 구조 확인

In [2]:
train_df = pd.read_parquet(TRAIN_PATH)
holdout_df = pd.read_parquet(HOLDOUT_PATH)

print(f'학습데이터: {train_df.shape}')
print(f'2025 홀드아웃: {holdout_df.shape}')
print()
print('학습에 있는데 홀드아웃엔 없는 컬럼:', sorted(set(train_df.columns) - set(holdout_df.columns)))
print('홀드아웃에 있는데 학습엔 없는 컬럼(참고, 문제 없음):', len(set(holdout_df.columns) - set(train_df.columns)), '개')


학습데이터: (6041091, 58)
2025 홀드아웃: (1612891, 96)

학습에 있는데 홀드아웃엔 없는 컬럼: ['1단계_대체수준_1', '1단계_대체수준_2', '매칭등급_A', '매칭등급_B', '매칭등급_C', '매칭등급_D']
홀드아웃에 있는데 학습엔 없는 컬럼(참고, 문제 없음): 44 개


## 2. 라벨 보정 (학습/홀드아웃 각각, 동일 로직)

In [3]:
def add_corrected_label(df, original_target_col):
    df = df.copy()
    df['최종_보정라벨'] = df[original_target_col]
    matched = df['인허가매칭성공여부'] == 1
    false_positive = (matched & (df['라벨_비교결과'] == '불일치_기존만폐업')
                       & df['폐업일자'].isna() & (df['영업상태명'] == '영업/정상'))
    df.loc[false_positive, '최종_보정라벨'] = 0
    false_negative = matched & (df['라벨_비교결과'] == '불일치_인허가만폐업')
    df.loc[false_negative, '최종_보정라벨'] = 1
    df['최종_보정라벨'] = df['최종_보정라벨'].astype('int8')
    n_changed = (df['최종_보정라벨'] != df[original_target_col]).sum()
    return df, n_changed


train_df, n1 = add_corrected_label(train_df, ORIGINAL_TARGET_COL)
holdout_df, n2 = add_corrected_label(holdout_df, ORIGINAL_TARGET_COL)
print(f'학습데이터 라벨 보정: {n1:,}건')
print(f'홀드아웃 라벨 보정: {n2:,}건')


학습데이터 라벨 보정: 26,062건
홀드아웃 라벨 보정: 4,484건


## 3. 홀드아웃 원-핫 인코딩 (매칭등급, 1단계_대체수준)

학습데이터는 이미 `매칭등급_A~D`, `1단계_대체수준_1/2`가 있어서 손 안 댐. 홀드아웃만 새로 생성.
카테고리 하나가 통째로 없을 경우(예: D등급이 홀드아웃에 아예 없음)를 대비해서 4개/2개 컬럼을
강제로 다 만들어둠 (없으면 0으로).

In [4]:
dummies_stage1 = pd.get_dummies(holdout_df['1단계_위험점수대체수준'], prefix='1단계_대체수준')
for col in ['1단계_대체수준_1', '1단계_대체수준_2']:
    if col not in dummies_stage1.columns:
        dummies_stage1[col] = 0
dummies_stage1 = dummies_stage1[['1단계_대체수준_1', '1단계_대체수준_2']].astype(int)

dummies_grade = pd.get_dummies(holdout_df['매칭등급'], prefix='매칭등급')
for col in ['매칭등급_A', '매칭등급_B', '매칭등급_C', '매칭등급_D']:
    if col not in dummies_grade.columns:
        dummies_grade[col] = 0
dummies_grade = dummies_grade[['매칭등급_A', '매칭등급_B', '매칭등급_C', '매칭등급_D']].astype(int)

holdout_df = pd.concat([holdout_df, dummies_stage1, dummies_grade], axis=1)
print('홀드아웃 원-핫 인코딩 완료')
print(dummies_stage1.sum())
print(dummies_grade.sum())


홀드아웃 원-핫 인코딩 완료
1단계_대체수준_1    0
1단계_대체수준_2    0
dtype: int64
매칭등급_A    252550
매칭등급_B     13704
매칭등급_C     72004
매칭등급_D      9840
dtype: int64


## 4. 홀드아웃을 학습데이터와 같은 컬럼 구조로 정렬

In [5]:
missing_in_holdout = [c for c in train_df.columns if c not in holdout_df.columns]
if missing_in_holdout:
    raise ValueError(f'홀드아웃에 없는 컬럼 발견, 원인 확인 필요: {missing_in_holdout}')

holdout_aligned = holdout_df[train_df.columns].copy()
print(f'정렬 완료: {holdout_aligned.shape} (학습데이터와 동일 컬럼 구조 {list(holdout_aligned.columns) == list(train_df.columns)})')

del holdout_df
gc.collect()


정렬 완료: (1612891, 59) (학습데이터와 동일 컬럼 구조 True)


0

## 5. 학습 + 홀드아웃 결합

In [6]:
train_df['_is_holdout'] = False
holdout_aligned['_is_holdout'] = True

combined = pd.concat([train_df, holdout_aligned], ignore_index=True)
combined = combined.sort_values([ID_COL, QUARTER_COL]).reset_index(drop=True)

del train_df, holdout_aligned
gc.collect()
print(f'결합 완료: {combined.shape}')
print(combined[QUARTER_COL].value_counts().sort_index())


결합 완료: (7653982, 60)
기준_년분기
2022Q1    524728
2022Q2    529441
2022Q3    534473
2022Q4    534294
2023Q1    474260
2023Q2    489903
2023Q3    482959
2023Q4    489872
2024Q1    484226
2024Q2    477378
2024Q3    466022
2024Q4    553535
2025Q1    540516
2025Q2    536114
2025Q3    536261
Name: count, dtype: int64


## 6. 신규 피처 10개 생성 (갭 안전 - v6과 동일 로직)

In [7]:
def quarter_add(q, delta):
    year = int(str(q)[:4]); qtr = int(str(q)[-1])
    total = year * 4 + (qtr - 1) + delta
    return f'{total // 4}Q{total % 4 + 1}'


def add_lagged_lookup(data, id_col, quarter_col, cols, lag):
    lookup = data[[id_col, quarter_col] + cols].rename(
        columns={quarter_col: '_lag_q', **{c: f'{c}_lag{lag}' for c in cols}}
    )
    key = pd.DataFrame({
        id_col: data[id_col].values,
        '_lag_q': data[quarter_col].map(lambda q: quarter_add(q, -lag)).values,
    })
    result = key.merge(lookup, on=[id_col, '_lag_q'], how='left')
    return result[[f'{c}_lag{lag}' for c in cols]].reset_index(drop=True)


def add_expanding_target_rate(data, group_col, quarter_col, target_col, out_col, smoothing_k=50):
    agg = data.groupby([group_col, quarter_col], observed=True)[target_col].agg(sum_='sum', count_='count').reset_index()
    agg = agg.sort_values([group_col, quarter_col])
    agg['cum_sum'] = agg.groupby(group_col, observed=True)['sum_'].cumsum() - agg['sum_']
    agg['cum_count'] = agg.groupby(group_col, observed=True)['count_'].cumsum() - agg['count_']
    global_mean = data[target_col].mean()
    agg[out_col] = (agg['cum_sum'] + smoothing_k * global_mean) / (agg['cum_count'] + smoothing_k)
    result = data[[group_col, quarter_col]].merge(agg[[group_col, quarter_col, out_col]], on=[group_col, quarter_col], how='left')
    return result[out_col].to_numpy()


# ---- 누적 타겟인코딩 ----
if '상권_코드' in combined.columns:
    combined['상권_누적_폐업률'] = add_expanding_target_rate(combined, '상권_코드', QUARTER_COL, TARGET_COL, '상권_누적_폐업률', smoothing_k=50)
if '서비스_업종_코드' in combined.columns:
    combined['업종_누적_폐업률'] = add_expanding_target_rate(combined, '서비스_업종_코드', QUARTER_COL, TARGET_COL, '업종_누적_폐업률', smoothing_k=50)

# ---- diff4 (전년동기대비) ----
DIFF4_COLS = ['1단계_위험점수', '반경100m_이웃수']
lag4 = add_lagged_lookup(combined, ID_COL, QUARTER_COL, DIFF4_COLS, lag=4)
for c in DIFF4_COLS:
    combined[f'{c}_diff4'] = (combined[c].to_numpy() - lag4[f'{c}_lag4'].to_numpy()).astype('float32')
del lag4

# ---- roll_std_4q ----
lag1 = add_lagged_lookup(combined, ID_COL, QUARTER_COL, ['1단계_위험점수'], lag=1)
lag2 = add_lagged_lookup(combined, ID_COL, QUARTER_COL, ['1단계_위험점수'], lag=2)
lag3 = add_lagged_lookup(combined, ID_COL, QUARTER_COL, ['1단계_위험점수'], lag=3)
lag4b = add_lagged_lookup(combined, ID_COL, QUARTER_COL, ['1단계_위험점수'], lag=4)
stacked = np.vstack([lag1['1단계_위험점수_lag1'].to_numpy(), lag2['1단계_위험점수_lag2'].to_numpy(),
                      lag3['1단계_위험점수_lag3'].to_numpy(), lag4b['1단계_위험점수_lag4'].to_numpy()])
with np.errstate(invalid='ignore'):
    combined['1단계_위험점수_roll_std_4q'] = np.nanstd(stacked, axis=0).astype('float32')
del lag1, lag2, lag3, lag4b, stacked
gc.collect()

# ---- 계절성 더미 ----
qtr_num = combined[QUARTER_COL].str[-1].astype(int)
for q in [2, 3, 4]:
    combined[f'분기_Q{q}여부'] = (qtr_num == q).astype('int8')

# ---- 상대비율 피처 ----
if '상권_누적_폐업률' in combined.columns and '업종_누적_폐업률' in combined.columns:
    combined['상권_대비_업종_폐업률_차이'] = (combined['상권_누적_폐업률'] - combined['업종_누적_폐업률']).astype('float32')

lag1n = add_lagged_lookup(combined, ID_COL, QUARTER_COL, ['반경100m_이웃수'], lag=1)
lag2n = add_lagged_lookup(combined, ID_COL, QUARTER_COL, ['반경100m_이웃수'], lag=2)
lag3n = add_lagged_lookup(combined, ID_COL, QUARTER_COL, ['반경100m_이웃수'], lag=3)
lag4n = add_lagged_lookup(combined, ID_COL, QUARTER_COL, ['반경100m_이웃수'], lag=4)
stacked_n = np.vstack([lag1n['반경100m_이웃수_lag1'].to_numpy(), lag2n['반경100m_이웃수_lag2'].to_numpy(),
                        lag3n['반경100m_이웃수_lag3'].to_numpy(), lag4n['반경100m_이웃수_lag4'].to_numpy()])
with np.errstate(invalid='ignore'):
    roll_mean_4q = np.nanmean(stacked_n, axis=0)
combined['반경100m_이웃수_대비_4분기평균비율'] = (combined['반경100m_이웃수'].to_numpy() / (roll_mean_4q + 1.0)).astype('float32')
del lag1n, lag2n, lag3n, lag4n, stacked_n, roll_mean_4q
gc.collect()

print(f'✅ 신규 피처 10개 생성 완료, 최종 shape: {combined.shape}')


✅ 신규 피처 10개 생성 완료, 최종 shape: (7653982, 70)


## 7. 다시 분리해서 저장

In [8]:
train_final = combined[~combined['_is_holdout']].drop(columns=['_is_holdout']).reset_index(drop=True)
holdout_final = combined[combined['_is_holdout']].drop(columns=['_is_holdout']).reset_index(drop=True)

del combined
gc.collect()

print(f'학습 최종: {train_final.shape}')
print(f'홀드아웃 최종: {holdout_final.shape}')

train_final.to_parquet(TRAIN_OUTPUT, index=False)
holdout_final.to_parquet(HOLDOUT_OUTPUT, index=False)
print(f'저장 완료: {TRAIN_OUTPUT}')
print(f'저장 완료: {HOLDOUT_OUTPUT}')


학습 최종: (6041091, 69)
홀드아웃 최종: (1612891, 69)
저장 완료: /content/drive/MyDrive/kb_challenge/2단계_학습데이터_최종_인허가포함_피처완료.parquet
저장 완료: /content/drive/MyDrive/kb_challenge/2단계_2025홀드아웃_피처완료.parquet


## 8. 검증 - 저장된 FEATURE_COLS(49개)가 둘 다 다 있는지 확인

In [9]:
SAVE_DIR = os.path.join(DRIVE_DIR.replace('kb_challenge', 'data'), 'saved_models_v6')
import joblib
FEATURE_COLS = joblib.load(os.path.join(SAVE_DIR, 'feature_cols.pkl'))

missing_train = [c for c in FEATURE_COLS if c not in train_final.columns]
missing_holdout = [c for c in FEATURE_COLS if c not in holdout_final.columns]

print('학습데이터에 없는 피처:', missing_train if missing_train else '없음 (정상)')
print('홀드아웃에 없는 피처:', missing_holdout if missing_holdout else '없음 (정상)')


학습데이터에 없는 피처: 없음 (정상)
홀드아웃에 없는 피처: 없음 (정상)
